
# 02 — Phân tích khám phá dữ liệu (EDA)

**Bộ dữ liệu:** Student Mental Health (Kaggle) — 101 bản ghi × 11 cột.

Notebook này dùng lại hàm `clean_raw_data` từ [`src/preprocessing.py`](../src/preprocessing.py) để
chuẩn hóa định dạng dữ liệu, rồi thực hiện 3 nhóm phân tích:

1. **Phân bố đơn biến** (Univariate) — phân bố từng biến.
2. **Mối liên hệ với biến mục tiêu** `Do you have Depression?` (Bivariate).
3. **Khảo sát triệu chứng đồng diễn** (Anxiety, Panic attack) — chỉ phục vụ ngữ cảnh,
   **không** đưa vào tập đặc trưng chính.

Toàn bộ biểu đồ được lưu dạng `.png` (dpi=300) vào `../results/figures/` và các bảng
thống kê chéo dạng `.csv` vào `../results/tables/` (đường dẫn được giải quyết từ thư mục
gốc repo để không phụ thuộc thư mục làm việc của kernel).


## 0. Thiết lập môi trường

In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Xác định thư mục gốc repo (chứa src/ và data/) bất kể cwd của kernel.
def _find_repo_root() -> Path:
    for cand in (Path.cwd(), Path.cwd().parent):
        if (cand / "src" / "preprocessing.py").exists():
            return cand
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc repo chứa src/preprocessing.py"
    )

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import clean_raw_data, get_feature_target_split

# Thư mục lưu artifact (tương đương ../results/figures và ../results/tables).
FIG_DIR = REPO_ROOT / "results" / "figures"
TABLE_DIR = REPO_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Cấu hình giao diện biểu đồ: nét rõ, tiêu đề đậm, có nhãn trục.
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TARGET = "Do you have Depression?"
TARGET_PALETTE = {"No": "#4C72B0", "Yes": "#C44E52"}

# Thứ tự hiển thị nhất quán cho các biến phân loại.
GENDER_ORDER = ["Female", "Male"]
YEAR_ORDER = ["year 1", "year 2", "year 3", "year 4"]
CGPA_ORDER = ["0 - 1.99", "2.00 - 2.49", "2.50 - 2.99", "3.00 - 3.49", "3.50 - 4.00"]
MARITAL_ORDER = ["No", "Yes"]

print("REPO_ROOT =", REPO_ROOT)
print("FIG_DIR   =", FIG_DIR)
print("TABLE_DIR =", TABLE_DIR)


In [ ]:

def save_fig(fig, name: str) -> None:
    '''Lưu biểu đồ dạng PNG (dpi=300) vào FIG_DIR.'''
    path = FIG_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"  [Lưu] {path}")


def save_table(df_table: pd.DataFrame, name: str) -> None:
    '''Lưu bảng thống kê dạng CSV vào TABLE_DIR.'''
    path = TABLE_DIR / name
    df_table.to_csv(path, encoding="utf-8")
    print(f"  [Lưu bảng] {path}")


def annotate_bars(ax, total=None, horizontal=False) -> None:
    '''Ghi số lượng (và tỷ lệ %) lên đầu các cột của biểu đồ.'''
    for p in ax.patches:
        if horizontal:
            x = p.get_width()
            if x <= 0:
                continue
            y = p.get_y() + p.get_height() / 2
            label = f"{x:.0f}"
            if total:
                label += f" ({100 * x / total:.1f}%)"
            ax.annotate(label, (x, y), ha="left", va="center", fontsize=9,
                        xytext=(3, 0), textcoords="offset points")
        else:
            x = p.get_x() + p.get_width() / 2
            y = p.get_height()
            if y <= 0:
                continue
            label = f"{y:.0f}"
            if total:
                label += f" ({100 * y / total:.1f}%)"
            ax.annotate(label, (x, y), ha="center", va="bottom", fontsize=9,
                        xytext=(0, 2), textcoords="offset points")


In [ ]:

RAW_PATH = REPO_ROOT / "data" / "raw" / "Student Mental health.csv"
raw = pd.read_csv(RAW_PATH, encoding="utf-8")
df = clean_raw_data(raw)              # 101 x 11 (đã strip + lower năm học)
X, y = get_feature_target_split(df)   # X: 6 feature, y: 0/1

print("Shape:", df.shape)
print("Số dòng trùng lặp hoàn toàn:", df.duplicated().sum())
print("\nGiá trị thiếu theo cột:")
print(df.isnull().sum()[lambda s: s > 0].to_string())

print("\nKiểm tra split từ preprocessing:")
print("X shape:", X.shape)
print("y phân bố:", y.value_counts().to_dict())

print("\nPhân bố biến mục tiêu:")
vc = df[TARGET].value_counts()
print(vc.to_frame("count")
      .assign(pct=lambda d: (d["count"] / len(df) * 100).round(2))
      .to_string())


## Nhóm 1: Phân bố đơn biến (Univariate analysis)

### 1.1 Biến mục tiêu `Do you have Depression?`

In [ ]:

vc = df[TARGET].value_counts()

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(data=df, x=TARGET, hue=TARGET, order=["No", "Yes"],
              palette=TARGET_PALETTE, ax=ax, legend=False)
annotate_bars(ax, total=len(df))
ax.set_title("Phân bố biến mục tiêu: Do you have Depression?")
ax.set_xlabel(TARGET)
ax.set_ylabel("Số lượng sinh viên")
save_fig(fig, "01_target_distribution.png")
plt.show()
plt.close(fig)

print("Số lượng:", vc.to_dict())
print("Tỷ lệ %:", (vc / len(df) * 100).round(2).to_dict())
print(f"Tỷ lệ No/Yes = {vc['No'] / vc['Yes']:.2f} : 1")


### 1.2 Giới tính, Năm học, Khoảng CGPA và Tình trạng hôn nhân

In [ ]:

cols = [
    ("Choose your gender", GENDER_ORDER),
    ("Your current year of Study", YEAR_ORDER),
    ("What is your CGPA?", CGPA_ORDER),
    ("Marital status", MARITAL_ORDER),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (col, order) in zip(axes.flat, cols):
    sns.countplot(data=df, x=col, order=order, color="#4C72B0", ax=ax)
    annotate_bars(ax, total=len(df))
    ax.set_title(f"Phân bố: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Số lượng sinh viên")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
save_fig(fig, "02_05_categorical_distributions.png")
plt.show()
plt.close(fig)

# Thống kê chi tiết từng biến
for col, _ in cols:
    vc = df[col].value_counts()
    print(f"\n--- {col} ---")
    print(vc.to_frame("count")
          .assign(pct=lambda d: (d["count"] / len(df) * 100).round(2))
          .to_string())


### 1.3 Tuổi (`Age`)

In [ ]:

age = df["Age"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(age.dropna(), kde=True, bins=range(17, 26), color="#4C72B0", ax=axes[0])
axes[0].set_title("Phân bố tuổi (Histogram + KDE)")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Số lượng")

sns.boxplot(x=age, color="#C44E52", ax=axes[1])
axes[1].set_title("Boxplot tuổi")
axes[1].set_xlabel("Age")

plt.tight_layout()
save_fig(fig, "06_age_distribution.png")
plt.show()
plt.close(fig)

q1, med, q3 = age.quantile([0.25, 0.5, 0.75])
print("mean    =", round(age.mean(), 2))
print("median  =", med)
print("min/max =", age.min(), "/", age.max())
print("Q1/Q3   =", q1, "/", q3)
print("IQR     =", round(q3 - q1, 2))
print("skew    =", round(age.skew(), 2))
print("Số giá trị thiếu =", int(age.isnull().sum()))


### 1.4 Top 10 ngành học phổ biến nhất (`What is your course?`)

In [ ]:

course_vc = df["What is your course?"].value_counts()
top10 = course_vc.head(10)
singletons = int((course_vc == 1).sum())

print(f"Số ngành học phân biệt (unique): {course_vc.size}")
print(f"Số ngành chỉ xuất hiện đúng 1 lần: {singletons}")

fig, ax = plt.subplots(figsize=(9, 6))
order = list(top10.index)
sns.countplot(data=df, y="What is your course?", order=order, color="#4C72B0", ax=ax)
annotate_bars(ax, total=len(df), horizontal=True)
ax.set_title("Top 10 ngành học phổ biến nhất")
ax.set_xlabel("Số lượng sinh viên")
ax.set_ylabel("Ngành học")
plt.tight_layout()
save_fig(fig, "07_course_top10.png")
plt.show()
plt.close(fig)

top10_df = top10.to_frame("count").assign(pct=lambda d: (d["count"] / len(df) * 100).round(2))
save_table(top10_df, "course_top10_frequency.csv")
print("\nBảng tần suất Top 10:")
print(top10_df.to_string())


## Nhóm 2: Mối liên hệ với biến mục tiêu `Do you have Depression?`

In [ ]:

def categorical_vs_target(col: str, fig_name: str, table_name: str, order=None) -> None:
    '''Vẽ cột nhóm + cột chồng 100%, lưu bảng chéo và in thống kê.'''
    ct = pd.crosstab(df[col], df[TARGET])
    if order:
        ct = ct.reindex([o for o in order if o in ct.index])
    row_pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)
    col_pct = ct.div(ct.sum(axis=0), axis=1).mul(100).round(1)

    # Lưu bảng chéo kết hợp số lượng + % theo hàng.
    table = ct.copy()
    table.columns = [f"{c}_count" for c in ct.columns]
    for c in ct.columns:
        table[f"{c}_pct_row"] = row_pct[c]
    table["total"] = ct.sum(axis=1)
    save_table(table, table_name)

    print(f"\n===== {col} vs {TARGET} =====")
    print("\n[1] Số lượng (crosstab):")
    print(ct.to_string())
    print(f"\n[2] Tỷ lệ % theo {col} (theo hàng):")
    print(row_pct.to_string())
    print("\n[3] Tỷ lệ % theo nhãn (theo cột):")
    print(col_pct.to_string())

    cat_order = list(ct.index)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # (a) Cột nhóm — số lượng.
    sns.countplot(data=df, x=col, hue=TARGET, order=cat_order,
                  hue_order=["No", "Yes"], palette=TARGET_PALETTE, ax=axes[0])
    axes[0].set_title(f"{col} theo {TARGET} (số lượng)")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Số lượng sinh viên")
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].legend(title=TARGET)
    for p in axes[0].patches:
        if p.get_height() > 0:
            axes[0].annotate(f"{p.get_height():.0f}",
                             (p.get_x() + p.get_width() / 2, p.get_height()),
                             ha="center", va="bottom", fontsize=9)

    # (b) Cột chồng 100%.
    ct_pct = ct.div(ct.sum(axis=1), axis=0).mul(100)
    ct_pct.plot(kind="bar", stacked=True, ax=axes[1],
                color=[TARGET_PALETTE["No"], TARGET_PALETTE["Yes"]])
    axes[1].set_title(f"Tỷ lệ % {TARGET} theo {col} (100%)")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Tỷ lệ %")
    axes[1].set_ylim(0, 100)
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].legend(title=TARGET, loc="upper right")
    for p in axes[1].patches:
        h = p.get_height()
        if h > 0:
            axes[1].annotate(f"{h:.1f}%",
                             (p.get_x() + p.get_width() / 2, p.get_y() + h / 2),
                             ha="center", va="center", fontsize=8,
                             color="white", fontweight="bold")

    plt.tight_layout()
    save_fig(fig, fig_name)
    plt.show()
    plt.close(fig)


### 2.1 Depression theo Giới tính

In [ ]:

categorical_vs_target(
    "Choose your gender",
    "08_gender_vs_depression.png",
    "crosstab_gender_depression.csv",
)


### 2.2 Depression theo Năm học

In [ ]:

categorical_vs_target(
    "Your current year of Study",
    "09_year_vs_depression.png",
    "crosstab_year_depression.csv",
    order=YEAR_ORDER,
)


### 2.3 Depression theo Khoảng CGPA

In [ ]:

categorical_vs_target(
    "What is your CGPA?",
    "10_cgpa_vs_depression.png",
    "crosstab_cgpa_depression.csv",
    order=CGPA_ORDER,
)


### 2.4 Depression theo Tình trạng hôn nhân

In [ ]:

categorical_vs_target(
    "Marital status",
    "11_marital_vs_depression.png",
    "crosstab_marital_depression.csv",
)


### 2.5 Tuổi theo nhóm Depression (Boxplot & Violin plot)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x=TARGET, y="Age", hue=TARGET,
            order=["No", "Yes"], hue_order=["No", "Yes"],
            palette=TARGET_PALETTE, legend=False, ax=axes[0])
axes[0].set_title("Boxplot Age theo Depression")
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel("Age")

sns.violinplot(data=df, x=TARGET, y="Age", hue=TARGET,
               order=["No", "Yes"], hue_order=["No", "Yes"],
               palette=TARGET_PALETTE, legend=False, ax=axes[1])
axes[1].set_title("Violin plot Age theo Depression")
axes[1].set_xlabel(TARGET)
axes[1].set_ylabel("Age")

plt.tight_layout()
save_fig(fig, "12_age_vs_depression.png")
plt.show()
plt.close(fig)

for grp in ["No", "Yes"]:
    sub = df.loc[df[TARGET] == grp, "Age"]
    print(f"\n--- Depression = {grp} ---")
    print(f"n={sub.notna().sum()}, mean={sub.mean():.2f}, median={sub.median():.1f}")
    print(f"Q1={sub.quantile(0.25):.1f}, Q3={sub.quantile(0.75):.1f}, "
          f"IQR={sub.quantile(0.75) - sub.quantile(0.25):.1f}")



## Nhóm 3: Khảo sát triệu chứng đồng diễn

> Chỉ phục vụ phân tích ngữ cảnh — các biến `Do you have Anxiety?` và
> `Do you have Panic attack?` **không** được đưa vào tập đặc trưng chính.


In [ ]:

symptoms = [("Do you have Anxiety?", "crosstab_depression_anxiety.csv"),
            ("Do you have Panic attack?", "crosstab_depression_panic_attack.csv")]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (sym, fname) in zip(axes, symptoms):
    ct = pd.crosstab(df[TARGET], df[sym])
    row_pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)

    print(f"\n===== {TARGET} vs {sym} =====")
    print("\nSố lượng (crosstab):")
    print(ct.to_string())
    print(f"\nTỷ lệ % theo hàng ({TARGET}):")
    print(row_pct.to_string())
    save_table(ct, fname)

    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"{TARGET} vs {sym}")
    ax.set_xlabel(sym)
    ax.set_ylabel(TARGET)

plt.tight_layout()
save_fig(fig, "13_comorbid_heatmaps.png")
plt.show()
plt.close(fig)



---
## Kết luận nhanh (phục vụ viết báo cáo)

- Dữ liệu mất cân bằng nhẹ: Depression `Yes` chiếm 34.65% (35/101).
- Các artifact đã lưu:
  - Biểu đồ: `results/figures/*.png` (10 file, dpi=300).
  - Bảng chéo: `results/tables/*.csv` (crosstab giới tính, năm học, CGPA, hôn nhân,
    anxiety, panic attack + tần suất top 10 ngành).
